In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()
import warnings
warnings.filterwarnings('ignore')
import datetime
import gensim
from gensim import corpora, models
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
import nltk
# nltk.download('wordnet')

In [2]:
dictionary_path = '/home/luxiaoling/wukun/MIND/data/large/train/gensim_dictionary'
lda_path = '/home/luxiaoling/wukun/MIND/data/large/train//lda_model'

In [3]:
dictionary = gensim.corpora.Dictionary.load_from_text(dictionary_path)
lda_model = gensim.models.LdaMulticore.load(lda_path)
topic_list=lda_model.print_topics(-1)
topic_list[163]

(163,
 '0.008*"shop" + 0.007*"dog" + 0.005*"royal" + 0.005*"famili" + 0.005*"like" + 0.005*"peopl" + 0.004*"love" + 0.004*"day" + 0.004*"time" + 0.004*"home"')

In [4]:
VAL_PATH = '/home/luxiaoling/wukun/MIND/data/large/dev/'
news_val_df = pd.read_csv(VAL_PATH+'news_with_body.tsv', sep='\t',
                         names=['news_id', 'category', 'subcategory', 'title', 'abstract', 'link', 'title_entity', 'abstract_entity', 'body'])

news_val_df.head()

,news_id,category,subcategory,title,abstract,link,title_entity,abstract_entity,body
0,N88753,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[],The royals are free to shop wherever they choo...
1,N23144,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","When you first start dieting and exercising, t..."
2,N86255,health,medical,Dispose of unwanted prescription drugs during ...,NaN,https://assets.msn.com/labs/mind/AAISxPN.html,"[{""Label"": ""Drug Enforcement Administration"", ...",[],"CINCINNATI (FOX19) - If you have expired, unus..."
3,N93187,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId...","ZOLOTE, Ukraine — Lt. Ivan Molchanets peeked o..."
4,N75236,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ...",I had to be perfect.In order to shed my perfec...


In [5]:
news_val_df.shape

(72020, 9)

In [6]:
def text2lower(text):
    try:
        return text.lower()
    except Exception as e:
        return '.'
news_val_df['title_lower'] = news_val_df['title'].apply(lambda x: x.lower())
news_val_df['abstract_lower'] = news_val_df['abstract'].apply(text2lower)
news_val_df['body_lower'] = news_val_df['body'].apply(text2lower)
news_val_df['title_abstract_body'] = news_val_df['title_lower'] + ['. '] * news_val_df.shape[0] + news_val_df['abstract_lower'] + ['. '] * news_val_df.shape[0] + news_val_df['body_lower']


In [7]:
stemmer = SnowballStemmer('english')
def lemmatize_stemming(text):
    return stemmer.stem(WordNetLemmatizer().lemmatize(text, pos='v'))
 
def preprocess(text):
    result = []
    for token in gensim.utils.simple_preprocess(text):
        if token not in gensim.parsing.preprocessing.STOPWORDS and len(token) > 2:
            result.append(lemmatize_stemming(token))
    return result

In [8]:
news_val_df['processed_title_abstract_body'] = news_val_df['title_abstract_body'].apply(preprocess)

In [9]:
def get_topic(processed_title_abstract_body):
    try:
        doc = dictionary.doc2bow(processed_title_abstract_body)
        topic_vector = lda_model[doc]
        topic_vector_sorted = sorted(topic_vector, key=lambda x: x[1], reverse=True)
        return topic_vector_sorted[0][0]
    except Exception as e:
        import random
        return random.randint(0, len(topic_list)-1)
get_topic(news_val_df['processed_title_abstract_body'][0])

163

In [10]:
import multiprocessing
multiprocessing.cpu_count()

16

In [11]:
# multiprocessing.Pool in jupyter notebook works on linux but not windows
pool = multiprocessing.Pool(multiprocessing.cpu_count())
news_val_df['topic'] = pool.map(get_topic, news_val_df['processed_title_abstract_body'])

In [12]:
from nltk.corpus import stopwords
stopwords = set(stopwords.words('english'))
import pandas as pd
import re
import numpy as np

In [13]:
def preprocess_for_w2v(text):
    words = re.sub(r'[!"#$%&\()*+,-./:;<=>?@\\^_`{|}~]', ' ', text).split()
    return [w for w in words if w not in stopwords and not w.isdigit()]

def text_clean(text):
    words = preprocess_for_w2v(text)
    words = [w if w in word2idx else 'UNK' for w in words]
    return ' '.join(words)


In [14]:
vocab_save_path = '/home/luxiaoling/wukun/MIND/data/large/vocab.json'
import json
with open(vocab_save_path, 'r') as f:
    word2idx = json.load(f)

In [15]:
news_val_df['clean_title'] = news_val_df['title_lower'].apply(text_clean)
news_val_df['clean_abstract'] = news_val_df['abstract_lower'].apply(text_clean)

In [30]:
news_val_df.head()

,news_id,category,subcategory,title,abstract,link,title_entity,abstract_entity,body,title_lower,abstract_lower,body_lower,title_abstract_body,processed_title_abstract_body,topic,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[],The royals are free to shop wherever they choo...,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",the royals are free to shop wherever they choo...,"the brands queen elizabeth, prince charles, an...","[brand, queen, elizabeth, princ, charl, princ,...",163,brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N23144,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","When you first start dieting and exercising, t...",50 worst habits for belly fat,these seemingly harmless habits are holding yo...,"when you first start dieting and exercising, t...",50 worst habits for belly fat. these seemingly...,"[worst, habit, belli, fat, seem, harmless, hab...",175,worst habits belly fat,seemingly harmless habits holding back keeping...
2,N86255,health,medical,Dispose of unwanted prescription drugs during ...,NaN,https://assets.msn.com/labs/mind/AAISxPN.html,"[{""Label"": ""Drug Enforcement Administration"", ...",[],"CINCINNATI (FOX19) - If you have expired, unus...",dispose of unwanted prescription drugs during ...,.,"cincinnati (fox19) - if you have expired, unus...",dispose of unwanted prescription drugs during ...,"[dispos, unwant, prescript, drug, dea, day, ci...",32,dispose unwanted prescription drugs dea's take...,
3,N93187,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId...","ZOLOTE, Ukraine — Lt. Ivan Molchanets peeked o...",the cost of trump's aid freeze in the trenches...,lt. ivan molchanets peeked over a parapet of s...,"zolote, ukraine — lt. ivan molchanets peeked o...",the cost of trump's aid freeze in the trenches...,"[cost, trump, aid, freez, trench, ukrain, war,...",104,cost trump's aid freeze trenches ukraine's war,lt ivan UNK peeked UNK sand bags front line wa...
4,N75236,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ...",I had to be perfect.In order to shed my perfec...,i was an nba wife. here's how it affected my m...,"i felt like i was a fraud, and being an nba wi...",i had to be perfect.in order to shed my perfec...,i was an nba wife. here's how it affected my m...,"[nba, wife, affect, mental, health, felt, like...",113,nba wife here's affected mental health,felt like fraud nba wife help fact nearly dest...


In [31]:
news_val_df.abstract_lower.isnull().sum()

0

In [32]:
news_val_df.clean_abstract.isnull().sum()

0

In [34]:
news_val_df.clean_abstract[2], news_val_df.abstract_lower[2]

('', '.')

In [35]:
def convert_null(text):
    if not text:
        return 'PAD'
    else:
        return text
news_val_df['clean_abstract'] = news_val_df['clean_abstract'].apply(convert_null)
news_val_df.head()

,news_id,category,subcategory,title,abstract,link,title_entity,abstract_entity,body,title_lower,abstract_lower,body_lower,title_abstract_body,processed_title_abstract_body,topic,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[],The royals are free to shop wherever they choo...,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",the royals are free to shop wherever they choo...,"the brands queen elizabeth, prince charles, an...","[brand, queen, elizabeth, princ, charl, princ,...",163,brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N23144,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","When you first start dieting and exercising, t...",50 worst habits for belly fat,these seemingly harmless habits are holding yo...,"when you first start dieting and exercising, t...",50 worst habits for belly fat. these seemingly...,"[worst, habit, belli, fat, seem, harmless, hab...",175,worst habits belly fat,seemingly harmless habits holding back keeping...
2,N86255,health,medical,Dispose of unwanted prescription drugs during ...,NaN,https://assets.msn.com/labs/mind/AAISxPN.html,"[{""Label"": ""Drug Enforcement Administration"", ...",[],"CINCINNATI (FOX19) - If you have expired, unus...",dispose of unwanted prescription drugs during ...,.,"cincinnati (fox19) - if you have expired, unus...",dispose of unwanted prescription drugs during ...,"[dispos, unwant, prescript, drug, dea, day, ci...",32,dispose unwanted prescription drugs dea's take...,PAD
3,N93187,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId...","ZOLOTE, Ukraine — Lt. Ivan Molchanets peeked o...",the cost of trump's aid freeze in the trenches...,lt. ivan molchanets peeked over a parapet of s...,"zolote, ukraine — lt. ivan molchanets peeked o...",the cost of trump's aid freeze in the trenches...,"[cost, trump, aid, freez, trench, ukrain, war,...",104,cost trump's aid freeze trenches ukraine's war,lt ivan UNK peeked UNK sand bags front line wa...
4,N75236,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ...",I had to be perfect.In order to shed my perfec...,i was an nba wife. here's how it affected my m...,"i felt like i was a fraud, and being an nba wi...",i had to be perfect.in order to shed my perfec...,i was an nba wife. here's how it affected my m...,"[nba, wife, affect, mental, health, felt, like...",113,nba wife here's affected mental health,felt like fraud nba wife help fact nearly dest...


In [36]:
dev_save_path = '/home/luxiaoling/wukun/MIND/data/large/dev/preprocessed_news.tsv'
news_val_df[['news_id', 'category', 'subcategory', 'topic', 'title_lower', 'abstract_lower', 'clean_title', 'clean_abstract']].to_csv(dev_save_path, sep='\t', header=False, index=False)


In [37]:
news_val_df.shape

(72020, 17)

In [2]:
train_news_path = '/home/luxiaoling/wukun/MIND/data/large/train/preprocessed_news.tsv'

In [3]:
train_news_df = pd.read_csv(train_news_path, sep='\t', names=['news_id', 'category', 'subcategory', 'topic', 'title_lower', 'abstract_lower', 'clean_title', 'clean_abstract'])
train_news_df.head()

,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,163,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N45436,news,newsscienceandtechnology,27,walmart slashes prices on last-generation ipads,apple's new ipad releases bring big deals on l...,walmart slashes prices last generation ipads,apple's new ipad releases bring big deals last...
2,N23144,health,weightloss,175,50 worst habits for belly fat,these seemingly harmless habits are holding yo...,worst habits belly fat,seemingly harmless habits holding back keeping...
3,N86255,health,medical,32,dispose of unwanted prescription drugs during ...,.,dispose unwanted prescription drugs dea's take...,PAD
4,N93187,news,newsworld,104,the cost of trump's aid freeze in the trenches...,lt. ivan molchanets peeked over a parapet of s...,cost trump's aid freeze trenches ukraine's war,lt ivan UNK peeked UNK sand bags front line wa...


In [4]:
train_news_df.subcategory.value_counts()

newsus                     14467
football_nfl               11812
newspolitics                5142
weathertopstories           4253
newscrime                   3676
                           ...  
travel-accessible              1
technology                     1
travel                         1
newstvmedia                    1
finance-insidetheticker        1
Name: subcategory, Length: 285, dtype: int64

In [5]:
category_list = list(train_news_df.subcategory.unique()) + ['Cate_UNK', 'Cate_PAD']
category_dict = dict(zip(category_list, range(len(category_list)))) 

In [6]:
len(category_dict)

287

In [7]:
import json
with open('/home/luxiaoling/wukun/MIND/data/large/cate2idx.json', 'w') as f:
    json.dump(category_dict, f, indent=4)

In [25]:
train_news_df.isnull().sum()

news_id              0
category             0
subcategory          0
topic                0
title_lower          0
abstract_lower    5415
clean_title          0
clean_abstract    5442
dtype: int64

In [26]:
train_news_df.fillna('.', inplace=True)

In [27]:
train_news_df.head()

,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,163,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N45436,news,newsscienceandtechnology,27,walmart slashes prices on last-generation ipads,apple's new ipad releases bring big deals on l...,walmart slashes prices last generation ipads,apple's new ipad releases bring big deals last...
2,N23144,health,weightloss,175,50 worst habits for belly fat,these seemingly harmless habits are holding yo...,worst habits belly fat,seemingly harmless habits holding back keeping...
3,N86255,health,medical,32,dispose of unwanted prescription drugs during ...,.,dispose unwanted prescription drugs dea's take...,.
4,N93187,news,newsworld,104,the cost of trump's aid freeze in the trenches...,lt. ivan molchanets peeked over a parapet of s...,cost trump's aid freeze trenches ukraine's war,lt ivan UNK peeked UNK sand bags front line wa...


In [28]:
def convert(text):
    if text == '.':
        return 'PAD'
    else:
        return text
train_news_df['clean_abstract'] = train_news_df['clean_abstract'].apply(convert)
train_news_df.head()

,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,163,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N45436,news,newsscienceandtechnology,27,walmart slashes prices on last-generation ipads,apple's new ipad releases bring big deals on l...,walmart slashes prices last generation ipads,apple's new ipad releases bring big deals last...
2,N23144,health,weightloss,175,50 worst habits for belly fat,these seemingly harmless habits are holding yo...,worst habits belly fat,seemingly harmless habits holding back keeping...
3,N86255,health,medical,32,dispose of unwanted prescription drugs during ...,.,dispose unwanted prescription drugs dea's take...,PAD
4,N93187,news,newsworld,104,the cost of trump's aid freeze in the trenches...,lt. ivan molchanets peeked over a parapet of s...,cost trump's aid freeze trenches ukraine's war,lt ivan UNK peeked UNK sand bags front line wa...


In [29]:
train_news_df.to_csv('./data/large/train/preprocessed_news.tsv', sep='\t', header=False, index=False)

In [2]:
## train_news & val_news 合并
train_news_path = '/home/luxiaoling/wukun/MIND/data/large/train/preprocessed_news.tsv'
val_news_path = '/home/luxiaoling/wukun/MIND/data/large/dev/preprocessed_news.tsv'

In [3]:
train_news_df = pd.read_csv(train_news_path, sep='\t', names=['news_id', 'category', 'subcategory', 'topic', 'title_lower', 'abstract_lower', 'clean_title', 'clean_abstract'])
train_news_df.head()

,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,163,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N45436,news,newsscienceandtechnology,27,walmart slashes prices on last-generation ipads,apple's new ipad releases bring big deals on l...,walmart slashes prices last generation ipads,apple's new ipad releases bring big deals last...
2,N23144,health,weightloss,175,50 worst habits for belly fat,these seemingly harmless habits are holding yo...,worst habits belly fat,seemingly harmless habits holding back keeping...
3,N86255,health,medical,32,dispose of unwanted prescription drugs during ...,.,dispose unwanted prescription drugs dea's take...,PAD
4,N93187,news,newsworld,104,the cost of trump's aid freeze in the trenches...,lt. ivan molchanets peeked over a parapet of s...,cost trump's aid freeze trenches ukraine's war,lt ivan UNK peeked UNK sand bags front line wa...


In [4]:
val_news_df = pd.read_csv(val_news_path, sep='\t', names=['news_id', 'category', 'subcategory', 'topic', 'title_lower', 'abstract_lower', 'clean_title', 'clean_abstract'])
val_news_df.head()


,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,163,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N23144,health,weightloss,175,50 worst habits for belly fat,these seemingly harmless habits are holding yo...,worst habits belly fat,seemingly harmless habits holding back keeping...
2,N86255,health,medical,32,dispose of unwanted prescription drugs during ...,.,dispose unwanted prescription drugs dea's take...,PAD
3,N93187,news,newsworld,104,the cost of trump's aid freeze in the trenches...,lt. ivan molchanets peeked over a parapet of s...,cost trump's aid freeze trenches ukraine's war,lt ivan UNK peeked UNK sand bags front line wa...
4,N75236,health,voices,113,i was an nba wife. here's how it affected my m...,"i felt like i was a fraud, and being an nba wi...",nba wife here's affected mental health,felt like fraud nba wife help fact nearly dest...


In [5]:
train_news_df.shape, val_news_df.shape

((101518, 8), (72020, 8))

In [6]:
news_df = pd.concat([train_news_df, val_news_df], axis=0)
news_df.shape

(173538, 8)

In [7]:
news_df.drop_duplicates(['news_id'], inplace=True)
news_df.shape

(104145, 8)

In [10]:
save_path = '/home/luxiaoling/wukun/MIND/data/large/preprocessed_all_news.tsv'
# news_df.to_csv(save_path, sep='\t', header=False, index=False)

In [11]:
all_news_df = pd.read_csv(save_path, sep='\t', names=['news_id', 'category', 'subcategory', 'topic', 'title_lower', 'abstract_lower', 'clean_title', 'clean_abstract'])
all_news_df.head()

,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
0,N88753,lifestyle,lifestyleroyals,163,"the brands queen elizabeth, prince charles, an...","shop the notebooks, jackets, and more that the...",brands queen elizabeth prince charles prince p...,shop notebooks jackets royals can't live without
1,N45436,news,newsscienceandtechnology,27,walmart slashes prices on last-generation ipads,apple's new ipad releases bring big deals on l...,walmart slashes prices last generation ipads,apple's new ipad releases bring big deals last...
2,N23144,health,weightloss,175,50 worst habits for belly fat,these seemingly harmless habits are holding yo...,worst habits belly fat,seemingly harmless habits holding back keeping...
3,N86255,health,medical,32,dispose of unwanted prescription drugs during ...,.,dispose unwanted prescription drugs dea's take...,PAD
4,N93187,news,newsworld,104,the cost of trump's aid freeze in the trenches...,lt. ivan molchanets peeked over a parapet of s...,cost trump's aid freeze trenches ukraine's war,lt ivan UNK peeked UNK sand bags front line wa...


In [13]:
all_news_df[all_news_df.news_id=='N110434']

,news_id,category,subcategory,topic,title_lower,abstract_lower,clean_title,clean_abstract
4663,N110434,lifestyle,lifestylefamilyandrelationships,103,the 50 most common last names in america,what's in a name?\thttps://assets.msn.com/labs...,common last names america,what's name https assets msn com labs mind UNK...
